# Reproduce — the single source of truth for results

Run this notebook top-to-bottom. It **recomputes every results-chapter table live** (Tables 3.1,
3.2, 3.3 — nothing from cache) and writes the one authoritative file **`results/RESULTS.md`**
(+ `results/all_numbers.csv`). Everything is explicitly seeded, so re-running gives identical numbers.

| Where | For |
|---|---|
| **this notebook / `results/RESULTS.md`** | the results (run it, or just read the file) |
| `walkthrough.ipynb` | a narrative explanation of the model + algorithms |
| `playground.ipynb` | tweak the grids and run searches yourself |

_Sampler: binomial (verified identical to the per-shot sampler, ~1000× faster) — the whole run is ~2 min._

In [1]:
import numpy as np, pandas as pd
from IPython.display import Markdown, display
import qmetrology as qm
from qmetrology import tables, conv95, config as C
pd.set_option('display.max_columns', None); pd.set_option('display.width', 220)
print('sampler :', qm.sim.DEFAULT_METHOD)
print('seeds   : tune', C.SEED_TUNE, '| test', C.SEED_TEST, '| unbiased', C.SEED_UNBIASED)
print('>95% grid sizes/column : linear', f'{conv95.grid_size("linear"):,}',
      '| binary', f'{conv95.grid_size("binary"):,}', '| reverse_eng', f'{conv95.grid_size("reverse_eng"):,}')

sampler : binomial
seeds   : tune 42 | test 2024 | unbiased 12345
>95% grid sizes/column : linear 6,912 | binary 7,776 | reverse_eng 1,152


## Recompute everything (~2 min)

In [2]:
res   = tables.compute(verbose=False)        # Table 3.1 col 1 + Table 3.3 (fixed budget 10,000)
res95 = tables.compute_conv95(verbose=False) # Table 3.1 col 2 + Table 3.2 (>95% budget)
tables.render_all(res, res95)                # -> results/RESULTS.md + results/all_numbers.csv
dfs = tables.dataframes(res, res95)

Wrote results/RESULTS.md and results/all_numbers.csv  (single source of truth)


### Table 3.1 — low precision (ε=1e-3, narrow): % converged @10k **and** >95% budget

In [3]:
dfs['t31']

,Algorithm,%conv paper,%conv honest,>95% paper,>95% honest
0,Separable (N=1),15.9,15.7,"100,000,000","995,238"
1,Brute force,56.3,56.2,"65,500","67,560"
2,Linear search,14.0,56.6,"66,707","72,077"
3,Binary search,11.5,47.5,"117,194","162,697"
4,Reverse Engineering,61.3,60.3,"87,976","107,575"


### Table 3.2 — >95%-convergence budget (this work / paper)

In [4]:
dfs['t32']

,Algorithm,"eps1e-3 U(0.01,0.1) honest","eps1e-3 U(0.01,0.1) paper","eps1e-4 U(0.01,0.1) honest","eps1e-4 U(0.01,0.1) paper","eps1e-4 U(0.001,0.01) honest","eps1e-4 U(0.001,0.01) paper","eps1e-4 U(0.001,0.1) honest","eps1e-4 U(0.001,0.1) paper"
0,Brute force,"67,560","65,500","7,820,925","6,450,000","670,390","628,000","6,589,005","5,591,385"
1,Linear search,"72,077","66,707","6,959,291","7,255,380","711,371","686,518","7,153,776","5,608,445"
2,Binary search,"162,697","117,194","10,335,954","9,062,907","2,637,335","1,372,431","20,543,470","10,590,461"
3,Reverse Engineering,"107,575","87,976","10,186,906","6,733,795","1,105,599","928,318","18,802,291","16,113,455"


### Table 3.3 — broad distributions: % converged @10k (this work vs paper)

In [5]:
dfs['t33']

,Distribution,Brute force paper,Brute force honest,Linear search paper,Linear search honest,Binary search paper,Binary search honest,Reverse Engineering paper,Reverse Engineering honest
0,"$\mathcal{U}(0.01, \pi/2)$",15.9,15.8,20.8,18.7,16.3,13.1,17.4,16.2
1,"$\mathcal{U}(0.01, \pi/4)$",20.4,22.1,24.4,24.4,21.3,18.4,22.3,25.1
2,"$\mathcal{U}(0.01, \pi/8)$",31.3,30.8,34.3,32.1,30.4,26.0,39.7,39.0
3,"$\mathcal{U}(0.01, \pi/16)$",43.3,42.5,46.6,43.0,39.3,37.0,51.8,50.2


## The written record — `results/RESULTS.md`
This is the single file to read/cite; it also contains the key findings and paste-ready LaTeX.

In [6]:
display(Markdown(open('results/RESULTS.md').read()))

# Results — single source of truth

_Regenerated by `reproduce.ipynb` (or `python -m qmetrology.reproduce`). Every number is **recomputed live** with explicit seeds — nothing is read from cache. `paper` = the published value; `this work` = recomputed honestly (tune on seed 42, validate on an independent seed 2024). Convergence rate = mean(|phi_hat - phi| < eps) over R=50,000 trials._

## Table 3.1 — low precision ($\epsilon=10^{-3}$, $\phi\sim\mathcal{U}(0.01,0.1)$)

| Algorithm | % converged @10k (paper) | % converged @10k (this work) | >95% budget (paper) | >95% budget (this work) |
|---|---|---|---|---|
| Separable (N=1) | 15.9% | 15.7% | 100,000,000 | 995,238 |
| Brute force | 56.3% | 56.2% | 65,500 | 67,560 |
| Linear search | 14.0% | 56.6% | 66,707 | 72,077 |
| Binary search | 11.5% | 47.5% | 117,194 | 162,697 |
| Reverse Engineering | 61.3% | 60.3% | 87,976 | 107,575 |
| _Reverse Eng. Lite_ | _paper 29.60% @ ~2000_ | _29.3% (m'=150, m=40)_ | — | _~3,875_ |

## Table 3.2 — budget for $>95\%$ convergence (this work / paper)

| Algorithm | eps1e-3 U(0.01,0.1) | eps1e-4 U(0.01,0.1) | eps1e-4 U(0.001,0.01) | eps1e-4 U(0.001,0.1) |
|---|---|---|---|---|
| Brute force | 67,560 / 65,500 (×1.03) | 7,820,925 / 6,450,000 (×1.21) | 670,390 / 628,000 (×1.07) | 6,589,005 / 5,591,385 (×1.18) |
| Linear search | 72,077 / 66,707 (×1.08) | 6,959,291 / 7,255,380 (×0.96) | 711,371 / 686,518 (×1.04) | 7,153,776 / 5,608,445 (×1.28) |
| Binary search | 162,697 / 117,194 (×1.39) | 10,335,954 / 9,062,907 (×1.14) | 2,637,335 / 1,372,431 (×1.92) | 20,543,470 / 10,590,461 (×1.94) |
| Reverse Engineering | 107,575 / 87,976 (×1.22) | 10,186,906 / 6,733,795 (×1.51) | 1,105,599 / 928,318 (×1.19) | 18,802,291 / 16,113,455 (×1.17) |

## Table 3.3 — broad distributions ($\epsilon=10^{-3}$, budget 10,000, % converged)

| Algorithm | $\mathcal{U}(0.01, \pi/2)$ | $\mathcal{U}(0.01, \pi/4)$ | $\mathcal{U}(0.01, \pi/8)$ | $\mathcal{U}(0.01, \pi/16)$ |
|---|---|---|---|---|
| Brute force | 15.8% (15.9%) | 22.1% (20.4%) | 30.8% (31.3%) | 42.5% (43.3%) |
| Linear search | 18.7% (20.8%) | 24.4% (24.4%) | 32.1% (34.3%) | 43.0% (46.6%) |
| Binary search | 13.1% (16.3%) | 18.4% (21.3%) | 26.0% (30.4%) | 37.0% (39.3%) |
| Reverse Engineering | 16.2% (17.4%) | 25.1% (22.3%) | 39.0% (39.7%) | 50.2% (51.8%) |

## Key findings

1. **Table 3.3 adaptive numbers are grid maxima (winner's curse).** In the paper each linear/binary/reverse-eng value is the *max* success over a large grid at only R=1000; de-biasing (re-check the winner at R=50k on a fresh seed) lowers them. Linear and reverse-engineering still beat brute force for broad distributions — the headline holds.
2. **Narrow Table 3.1 linear (14.0%) & binary (11.5%) do not reproduce** (this work: linear 57%, binary 48%). They look stale; the best achievable at budget 10,000 is much higher.
3. **Separable's >95% budget of 100,000,000 is ~100× too high** — honest ~995,238 (SQL scaling: brute 65,500 × N_min=15 ≈ 1M).
4. **>95% budgets:** brute force is the reliable anchor (≈paper). The adaptive budgets are somewhat optimistic — **binary the most (×1.3–1.9)**, linear/reverse-eng ×1.0–1.5.
5. **Reverse-Engineering Lite's '29.60% @ ~2000' is not one config**: ~29.6% needs m'≈150–200 (budget ~3,800); ~2,000 budget gives ~22%.

## Methodology & reproducibility

- **Sampler:** binomial (`hits ~ Binomial(m, cos²Nφ)`) — verified identical to the original per-shot sampler (`analysis/confirm_binomial.py`), ~1000× faster for large m, so the whole run (Tables 3.1/3.2/3.3) takes ~2 min including ε=1e-4.
- **Fair >95% protocol:** grid-tune on seed 42, then report the smallest budget whose *validated* success ≥95% on seed 2024 (avoids the winner's curse).
- **Grid resolution (one knob per axis, both geomspace):** `conv95.N_EXPLORE=12`, `conv95.N_EXPLOIT=24` → linear 6,912, binary 7,776, reverse-eng 1,152 configs per column. Edit these (or use `playground.ipynb`) to search harder.
- **Seeds:** SEED_TUNE=42, SEED_TEST=2024, SEED_UNBIASED=12345 — re-running gives identical numbers.
- _The repo's bundled `main-thesis.pdf` shows Table 3.2 col 3 as ε=1e-3; the author's current version has ε=1e-4 (used here), which the printed budgets match._

## LaTeX (paste-ready, honest numbers)

**Table 3.1:**
```latex
\begin{tabular}{lcccc}
\toprule
 & \multicolumn{2}{c}{\% converged @ budget 10\,000} & \multicolumn{2}{c}{Budget for $>$95\% conv.} \\
Algorithm & paper & this work & paper & this work \\
\midrule
Separable (N=1) & 15.9\% & 15.7\% & 100,000,000 & 995,238 \\
Brute force & 56.3\% & 56.2\% & 65,500 & 67,560 \\
Linear search & 14.0\% & 56.6\% & 66,707 & 72,077 \\
Binary search & 11.5\% & 47.5\% & 117,194 & 162,697 \\
Reverse Engineering & 61.3\% & 60.3\% & 87,976 & 107,575 \\
\bottomrule
\end{tabular}
```

**Table 3.3:**
```latex
\begin{tabular}{lcccc}
\toprule
Algorithm & $\mathcal{U}(0.01, \pi/2)$ & $\mathcal{U}(0.01, \pi/4)$ & $\mathcal{U}(0.01, \pi/8)$ & $\mathcal{U}(0.01, \pi/16)$ \\
\midrule
Brute force & 15.8\% & 22.1\% & 30.8\% & 42.5\% \\
Linear search & 18.7\% & 24.4\% & 32.1\% & 43.0\% \\
Binary search & 13.1\% & 18.4\% & 26.0\% & 37.0\% \\
Reverse Engineering & 16.2\% & 25.1\% & 39.0\% & 50.2\% \\
\bottomrule
\end{tabular}
```
